In [1]:
import pysftp
import sys
import os
import pandas as pd

c:\Users\jonas\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2024'

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [4]:
cnopts = pysftp.CnOpts()
cnopts.hostkeys = None

c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jonas\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


In [5]:
dir_out = "../parsed_data/"

In [6]:
map_bz_country = {'AL' : 'AL',
				  'AT' : 'AT',
				  'BA' : 'BA',
				  'BE' : 'BE',
				  'BG' : 'BG',
				  'BY' : 'BY',
				  'CH' : 'CH',
				  'CZ' : 'CZ',
				  'DE' : 'DE',
				  'DE_50HzT' : 'DE',
				  'DE_Amprion' : 'DE',
				  'DE_TenneT_GER' : 'DE',
				  'DE_TransnetBW' : 'DE',
				  'DK' : 'DK',
				  'DK1' : 'DK',
				  'DK2' : 'DK',
				  'EE' : 'EE',
				  'ES' : 'ES',
				  'FI' : 'FI',
				  'FR' : 'FR',
				  'GB' : 'GB',
				  'GR' : 'GR',
				  'HR' : 'HR',
				  'HU' : 'HU',
				  'IE' : 'IE',
				  'IE_SEM' : 'IE',
				  'IT' : 'IT',
				  'IT_BRNN' : 'IT',
				  'IT_CNOR' : 'IT',
				  'IT_CSUD' : 'IT',
				  'IT_FOGN' : 'IT',
				  'IT_GR' : 'IT',
				  'IT_NORD_AT' : 'IT',
				  'IT_NORD_CH' : 'IT',
				  'IT_NORD_FR' : 'IT',
				  'IT_North' : 'IT',
				  'IT_PRGP' : 'IT',
				  'IT_ROSN' : 'IT',
				  'IT_SACO_AC' : 'IT',
				  'IT_SACO_DC' : 'IT',
				  'IT_SARD' : 'IT',
				  'IT_SICI' : 'IT',
				  'IT_SUD' : 'IT',
				  'LT' : 'LT',
				  'LU' : 'LU',
				  'LV' : 'LV',
				  'ME' : 'ME',
				  'MK' : 'MK',
				  'MT' : 'MT',
				  'NIE' : 'NI',
				  'NL' : 'NL',
				  'NO' : 'NO',
				  'NO1' : 'NO',
				  'NO2' : 'NO',
				  'NO3' : 'NO',
				  'NO4' : 'NO',
				  'NO5' : 'NO',
				  'PL' : 'PL',
				  'PT' : 'PT',
				  'RO' : 'RO',
				  'RS' : 'RS',
				  'RU' : 'RU',
				  'RU_KGD' : 'RU',
				  'SE' : 'SE',
				  'SE1' : 'SE',
				  'SE2' : 'SE',
				  'SE3' : 'SE',
				  'SE4' : 'SE',
				  'SI' : 'SI',
				  'SK' : 'SK',
				  'TR' : 'TR',
				  'UA' : 'UA',
				  'UA_BEI' : 'UA',
				  'UA_DobTPP' : 'UA'
}

In [7]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [8]:
with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
    print("Connection succesfully established.")

    # show list of files
    files = sftp.listdir('/TP_export/')   
    print(files)

Connection succesfully established.
['AcceptedAggregatedOffers_17.1.D', 'ActivatedBalancingEnergy_17.1.E', 'ActualCapacitiesAndOutlookOnFrequencyRestorationReserveAndReplacementReserve_SOGL_188.3_188.4_189.2_189.3_r3', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r2.1', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r3', 'ActualTotalLoad_6.1.A', 'ActualTotalLoad_6.1.A_r3', 'AggregatedBalancingEnergyBids_12.3.E_r3', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D_r3', 'AggregatedGenerationPerType_16.1.B_C', 'AggregatedGenerationPerType_16.1.B_C_r3', 'AmountAndPricesPaidOfBalancingReservesUnderContract_17.1.B_C_r3', 'AmountOfBalancingReservesUnderContract_17.1.B', 'AuctionRevenue_12.1.A_r3', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r3', 'ChangesInActualAvailabilityOfConsumptionUnits_7.1.B', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructureReasons_10.1.C', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructure_10.1.C', 'ChangesToBidAvailability_IFs_

## load data

In [9]:
#set paths and get file names
path_trade = path+'CommercialSchedules_12.1.F_r3/'
path_trade_local = path_local+'trade/'
with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_trade)
    #download files
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [10]:
#download aggregated data
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_trade+file,path_trade_local+file)
            print('Successfully downloaded file '+file)

In [11]:
df_temp = pd.read_csv(path_trade_local+files[1],sep="\t")
df_temp.head()

,DateTime(UTC),ResolutionCode,OutAreaCode,OutAreaDisplayName,OutAreaTypeCode,OutMapCode,InAreaCode,InAreaDisplayName,InAreaTypeCode,InMapCode,DayAheadCapacity[MW],TotalCapacity[MW],UpdateTime(UTC)
0,2024-02-01 00:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,356.0,356.0,2025-09-30 05:40:16
1,2024-02-01 01:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,375.0,375.0,2025-09-30 05:40:16
2,2024-02-01 02:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,342.0,342.0,2025-09-30 05:40:16
3,2024-02-01 03:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,359.0,359.0,2025-09-30 05:40:16
4,2024-02-01 04:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,400.0,400.0,2025-09-30 05:40:16


In [13]:
df_temp = pd.read_csv(path_trade_local+files[1],
                          sep="\t",
                          parse_dates=True, index_col="DateTime(UTC)")
df_temp.head()

,ResolutionCode,OutAreaCode,OutAreaDisplayName,OutAreaTypeCode,OutMapCode,InAreaCode,InAreaDisplayName,InAreaTypeCode,InMapCode,DayAheadCapacity[MW],TotalCapacity[MW],UpdateTime(UTC)
DateTime(UTC),,,,,,,,,,,,
2024-02-01 00:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,356.0,356.0,2025-09-30 05:40:16
2024-02-01 01:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,375.0,375.0,2025-09-30 05:40:16
2024-02-01 02:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,342.0,342.0,2025-09-30 05:40:16
2024-02-01 03:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,359.0,359.0,2025-09-30 05:40:16
2024-02-01 04:00:00,PT60M,10YAL-KESH-----5,Albania (AL),BZN/CTA/CTY,AL,10YGR-HTSO-----Y,Greece (GR),BZN/CTA/CTY,GR,400.0,400.0,2025-09-30 05:40:16


In [15]:
#combine files to one data frame
df_trade = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_trade_local+file,
                          sep="\t",
                          parse_dates=True, index_col="DateTime(UTC)")
    df_trade = pd.concat([df_trade,df_temp])
df_trade = df_trade.drop(['ResolutionCode','OutAreaDisplayName','OutAreaCode','InAreaCode','InAreaTypeCode','InAreaDisplayName','UpdateTime(UTC)'], axis=1).reset_index()
df_trade = df_trade.sort_values(by=['DateTime(UTC)'])
df_trade.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4032274 entries, 0 to 3857655
Data columns (total 6 columns):
 #   Column                Dtype         
---  ------                -----         
 0   DateTime(UTC)         datetime64[ns]
 1   OutAreaTypeCode       object        
 2   OutMapCode            object        
 3   InMapCode             object        
 4   DayAheadCapacity[MW]  float64       
 5   TotalCapacity[MW]     float64       
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 215.3+ MB


In [16]:
df_trade = df_trade.rename(columns={'DateTime(UTC)':'time','OutAreaTypeCode':'type','OutMapCode':'from_country','InMapCode':'to_country','TotalCapacity[MW]':'MWh'})
df_trade = df_trade.set_index(['type','from_country','to_country','time'])
df_trade.head()

DayAheadCapacity[MW]     MWh
type        from_country to_country time                                    
BZN/CTA/CTY AL           GR         2024-01-01                 348.0   348.0
            FR           GB         2024-01-01                3014.0  3014.0
BZN/CTA     UA_IPS       BY         2024-01-01                   0.0     0.0
BZN/CTA/CTY ME           IT         2024-01-01                 422.0   422.0
BZN         NO2          NO5        2024-01-01                   0.0     0.0

In [17]:
df_trade_pivot =  df_trade.pivot_table(index=['from_country','to_country','time'],columns='type',values='MWh').reset_index()
df_trade_pivot.from_country = df_trade_pivot.from_country.map(map_bz_country)
df_trade_pivot.to_country = df_trade_pivot.to_country.map(map_bz_country)
df_trade_pivot = df_trade_pivot.groupby(['from_country','to_country','time']).sum()
df_trade_pivot['MWh'] = df_trade_pivot.max(axis=1)
df_trade_pivot = df_trade_pivot[['MWh']]
df_trade_pivot.head()

type                                           MWh
from_country to_country time                      
AL           GR         2024-01-01 00:00:00  348.0
                        2024-01-01 01:00:00  312.0
                        2024-01-01 02:00:00  261.0
                        2024-01-01 03:00:00  312.0
                        2024-01-01 04:00:00  358.0

In [18]:
df_trade_pivot.to_csv(dir_out+'trade_'+year+'_hourly_entsoe.csv', encoding="utf-8")